<a href="https://colab.research.google.com/github/AKugeleviciute/DeepLearning/blob/main/individual_task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Natural language model**

**Goal:**

Build a semantic search tool for a selected context using transformer-based natural language models as feature extractors, and store the resulting vector representations in a vector database.

**Steps:**

**1.   Data Collection:**
scrape a website and collect at least 5,000 data entries.

**2.   Feature Extraction:**
use a transformer model to generate embeddings and save them to a vector database.

**3.   Semantic Search:**
implement a search function that processes queries, retrieves relevant entries via vector similarity, and uses Retrieval Augmented Generation (RAG) for improved results.


**Author**: Aina Kugelevičiūtė





# Web scraping

 At least 5,000 data entries will be collected by scraping quotes from the website https://www.goodreads.com/quotes?page={}.

*Reference:* https://codedamn.com/news/python/web-scraping-using-python

In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
tag = "inspirational-quotes"
url = f"https://www.goodreads.com/quotes/tag/{tag}"

# Headers are used to mimic a real web browser (some websites block requests that don't have a user-agent string)
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Send a GET request to the URL with the headers to get the page content
res = requests.get(url, headers=headers)

# Parse the response content (HTML) using BeautifulSoup
soup = BeautifulSoup(res.text, "html.parser")

In [3]:
# Find all div elements that have the class attribute "quote"
quote_divs = soup.find_all("div", attrs={"class": "quote"})

print(quote_divs)

[<div class="quote mediumText">
<div class="quoteDetails">
<a class="leftAlignedImage quoteAvatar" href="/author/show/4039811.Veronica_Roth">
<img alt="Veronica Roth" src="https://i.gr-assets.com/images/S/compressed.photo.goodreads.com/authors/1729525649i/4039811._UX200_CR0,45,200,200_.jpg"/>
</a>
<div class="quoteText">
      “We believe in ordinary acts of bravery, in the courage that drives one person to stand up for another.”
    <br/>
  ―
  <span class="authorOrTitle">
    Veronica Roth,
  </span>
<span id="quote_book_link_8306857">
<a class="authorOrTitle" href="/work/quotes/13155899">Divergent</a>
</span>
</div>
<div class="quoteFooter">
<div class="greyText smallText left">
     tags:
       <a href="/quotes/tag/inspirational-quotes">inspirational-quotes</a>,
       <a href="/quotes/tag/strength-and-courage">strength-and-courage</a>
</div>
<div class="right">
<a class="smallText" href="/quotes/378643-we-believe-in-ordinary-acts-of-bravery-in-the-courage" title="View this quote"

In [4]:
# Select the first div element with the class 'quote' from the list of quotes
quote_div = quote_divs[0]

# Find the first div element with the class 'quoteText' inside the selected 'quote' div element
quoteText_div = quote_div.find_next("div", attrs={"class": "quoteText"})
print(quoteText_div)

<div class="quoteText">
      “We believe in ordinary acts of bravery, in the courage that drives one person to stand up for another.”
    <br/>
  ―
  <span class="authorOrTitle">
    Veronica Roth,
  </span>
<span id="quote_book_link_8306857">
<a class="authorOrTitle" href="/work/quotes/13155899">Divergent</a>
</span>
</div>


In [5]:
# Format the text
striped = quoteText_div.text.strip()
striped_text = striped.split("\n")

# Extract only the quote text and author's name
quote = striped_text[0][1:-1]
author = striped_text[-1].strip()

print(quote)
print(author)

We believe in ordinary acts of bravery, in the courage that drives one person to stand up for another.
Divergent


In [6]:
# Find the next div element that has the class attribute "greyText smallText left" inside the div element with the class "quote"
tags_div = quote_div.find_next("div", attrs={"class" : "greyText smallText left"})

print(tags_div)

<div class="greyText smallText left">
     tags:
       <a href="/quotes/tag/inspirational-quotes">inspirational-quotes</a>,
       <a href="/quotes/tag/strength-and-courage">strength-and-courage</a>
</div>


In [7]:
# Extract tags
tags = [tag.strip() for tag in tags_div.text.split("tags:")[1].split(",")]

print(tags)

['inspirational-quotes', 'strength-and-courage']


In [8]:
def getAllQuotes():
    quotes = []
    base_url = "https://www.goodreads.com/quotes/tag/inspirational-quotes?page={}"
    headers = {"User-Agent": "Mozilla/5.0"}

    for i in range(1, 21): # iterate over 20 pages
        page_url = base_url.format(i)
        res = requests.get(page_url, headers=headers)
        soup = BeautifulSoup(res.text, "html.parser")

        # Find all div elements with the class "quote"
        quote_divs = soup.find_all("div", class_="quote")
        for quote_div in quote_divs:
            quoteText_div = quote_div.find("div", class_="quoteText")
            if quoteText_div is None:
                continue

            # Quote and author extraction:
            striped_text = quoteText_div.text.strip().split("\n")
            quote = striped_text[0].strip('“”"').strip()
            author = striped_text[-1].strip()

            # Tags extraction
            tags_div = quote_div.find("div", class_="greyText smallText left")
            if tags_div:
                tags_text = tags_div.text.strip()
                tags = [t.strip() for t in tags_text.split("tags:")[-1].split(",")]
            else:
                tags = []

            # Store the quote data
            quote_item = {"quote": quote, "author": author, "tags": tags}
            quotes.append(quote_item)

    return quotes

In [9]:
quote_data = getAllQuotes()

# Print the number of quotes collected
print(f"Collected {len(quote_data)} quotes.")

# Calculate total word count of all collected quotes
total_words = sum(len(quote['quote'].split()) for quote in quote_data)
print(f"Total number of words in all quotes: {total_words}")

print(quote_data)

Collected 600 quotes.
Total number of words in all quotes: 21396
[{'quote': 'We believe in ordinary acts of bravery, in the courage that drives one person to stand up for another.', 'author': 'Divergent', 'tags': ['inspirational-quotes', 'strength-and-courage']}, {'quote': "Don't think or judge, just listen.", 'author': 'Just Listen', 'tags': ['inspirational-quotes']}, {'quote': "Don't be pushed around by the fears in your mind. Be led by the dreams in your heart.", 'author': 'The Light in the Heart', 'tags': ['dreams', 'fearless', 'heart', 'inspiration', 'inspirational', 'inspirational-attitude', 'inspirational-life', 'inspirational-quotes', 'inspire', 'inspiring', 'life', 'life-and-living', 'life-lessons', 'life-quotes', 'living', 'mind', 'motivation', 'motivational', 'optimism', 'optimistic', 'positive', 'positive-affirmation', 'positive-life', 'positive-thinking']}, {'quote': 'Instead of worrying about what you cannot control, shift your energy to what you can create.', 'author': '

# Data

In [10]:
import pandas as pd

In [11]:
df = pd.DataFrame(quote_data)

# View first lines
df.head()

,quote,author,tags
0,"We believe in ordinary acts of bravery, in the...",Divergent,"[inspirational-quotes, strength-and-courage]"
1,"Don't think or judge, just listen.",Just Listen,[inspirational-quotes]
2,Don't be pushed around by the fears in your mi...,The Light in the Heart,"[dreams, fearless, heart, inspiration, inspira..."
3,Instead of worrying about what you cannot cont...,The Light in the Heart,"[creation, creative, inspiration, inspirationa..."
4,"You can talk with someone for years, everyday,...",C. JoyBell C.,"[affection, communication, heartwarming, human..."


In [12]:
print("Data shape:", df.shape)

Data shape: (600, 3)


In [13]:
# Convert to CSV
df.to_csv("scrap.csv", index = None)

# Model

In [14]:
!pip install transformers

In [15]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import torch

In [16]:
from transformers import BertTokenizer, BertModel

In [17]:
# Load the tokenizer from HuggingFace's Transformers library
# 'bert-base-uncased' is a pre-trained BERT model with lowercase-only vocabulary
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load the corresponding BERT model
# Setting 'output_hidden_states=True' allows us to access internal layer representations
model = BertModel.from_pretrained("bert-base-uncased", output_hidden_states=True)

# Specify the computation device — using CPU in this case
device = torch.device("cpu")
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [18]:
# Tokenize the list of quotes from the dataframe
# - return_tensors='pt' returns PyTorch tensors
# - padding=True pads sequences to the same length
# - truncation=True cuts off long sequences beyond max_length
# - max_length=32 ensures each input sequence is no longer than 32 tokens
encoded_inputs = tokenizer(df['quote'].tolist(), return_tensors='pt',
                           padding=True, truncation=True, max_length=32)

# Generate BERT embeddings
with torch.no_grad():
    model.eval()
    encoded_inputs = encoded_inputs.to(device)
    embeddings = model(**encoded_inputs)[2][-1][:, 0, :].cpu().numpy()

# Calculate pairwise cosine similarities between all quote embeddings
similarities = cosine_similarity(embeddings)

In [19]:
## Find the top 5 most similar sentences to the first quote
top_k  = 5

# Get similarity scores for the first quote (index 0)
similar_indices = np.argsort(similarities[0])[::-1][1:top_k + 1]

# Retrieve the actual quotes from the dataframe based on the indices
similar_quotes = df.iloc[similar_indices]['quote'].tolist()

# Extract the most similar quotes and their scores
print(f"Top {top_k} most similar sentences to '{df['quote'][0]}'")
for i, sentence in enumerate(similar_quotes):
    similarity_score = similarities[0][similar_indices[i]]
    print(f"{i+1}. '{sentence}' (similarity score: {similarity_score:.3f})")

Top 5 most similar sentences to 'We believe in ordinary acts of bravery, in the courage that drives one person to stand up for another.'
1. 'Unconditional love is the greatest gift we can ever give.' (similarity score: 0.916)
2. 'It takes guts and humility to admit mistakes. Admitting we're wrong is courage, not weakness.' (similarity score: 0.914)
3. 'We follow the codes not because they bring gain, but because we loathe the people we would otherwise become.' (similarity score: 0.914)
4. 'It's not our job to toughen our children up to face a cruel and heartless world. It's our job to raise children who will make the world a little less cruel and heartless.' (similarity score: 0.905)
5. 'What helps you persevere is your resilience and commitment.' (similarity score: 0.900)


In [20]:
## Find the top 5 most similar sentences for each sentence
top_k  = 5
all_most_similar  = []

# Iterate over each quote in the dataset
for i in range(len(similarities)):
    # Exclude the quote itself (self-similarity)
    top_indices = np.argsort(similarities[i])[::-1][1:top_k+1]
    top_quotes = df.iloc[top_indices]['quote'].tolist()
    all_most_similar.append(top_quotes)

# Display the top similar quotes for each quote
for i, similar_quotes in enumerate(all_most_similar):
    original_quote = df.loc[i, 'quote']
    print(f"\nTop {top_k} most similar quotes to:\n\"{original_quote}\"\n")

    top_indices = np.argsort(similarities[i])[::-1][1:top_k+1]
    for rank, idx in enumerate(top_indices, 1):
        quote = df.iloc[idx]['quote']
        score = similarities[i][idx]
        print(f"{rank}. \"{quote}\" (Similarity Score: {score:.3f})")


Top 5 most similar quotes to:
"We believe in ordinary acts of bravery, in the courage that drives one person to stand up for another."

1. "Unconditional love is the greatest gift we can ever give." (Similarity Score: 0.916)
2. "It takes guts and humility to admit mistakes. Admitting we're wrong is courage, not weakness." (Similarity Score: 0.914)
3. "We follow the codes not because they bring gain, but because we loathe the people we would otherwise become." (Similarity Score: 0.914)
4. "It's not our job to toughen our children up to face a cruel and heartless world. It's our job to raise children who will make the world a little less cruel and heartless." (Similarity Score: 0.905)
5. "What helps you persevere is your resilience and commitment." (Similarity Score: 0.900)

Top 5 most similar quotes to:
"Don't think or judge, just listen."

1. "Don't be pushed around by the fears in your mind. Be led by the dreams in your heart." (Similarity Score: 0.919)
2. "Be where you are, stop ove

In [21]:
new = "Deep learning"

# Combine existing quotes with the new input for joint embedding
all_sentences = df['quote'].tolist() + [new]

# Tokenize all sentences, including the new one
encoded_inputs = tokenizer(all_sentences, return_tensors='pt', padding=True, truncation=True, max_length=32)

# Generate embeddings using the BERT model
with torch.no_grad():
    model.eval()
    encoded_inputs = encoded_inputs.to(device)
    embeddings = model(**encoded_inputs)[2][-1][:, 0, :].cpu().numpy()

# Compute cosine similarity between all sentence embeddings
similarities = cosine_similarity(embeddings)

# Identify the index of the new input (last in the list)
new_index = len(all_sentences) - 1

# Find the top 5 most similar sentences for the new quote
n_similar = 5
similar_indices = np.argsort(similarities[new_index])[::-1][1:n_similar+1]
similar_sentences = np.array(all_sentences)[similar_indices]
similarity_scores = similarities[new_index][similar_indices]


print(f"Top {n_similar} most similar sentences to '{new}'")
for i, (sentence, score) in enumerate(zip(similar_sentences, similarity_scores)):
    print(f"{i+1}. {sentence} (similarity score: {score:.3f})")

Top 5 most similar sentences to 'Deep learning'
1. My Life is My Message (similarity score: 0.950)
2. Push your boundaries beyond the ordinary; be that “extra” in “extraordinary. (similarity score: 0.927)
3. Never underestimate a bookworm (similarity score: 0.924)
4. Be the change you want to see in the world (similarity score: 0.891)
5. I have a dream.” “Is your dream very big?” “No.” “Well is it very grand?” “Not either.” “Neither?” “Neither.” “What is it then?” “It is very shiny. (similarity score: 0.886)


# Retrieval Augmented Generation (RAG)

In [22]:
!pip install faiss-cpu sentence-transformers transformers

In [23]:
from sentence_transformers import SentenceTransformer
import faiss

from transformers import pipeline

In [24]:
# Load CSV
df = pd.read_csv("scrap.csv")

# Use Sentence-BERT for sentence embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
quote_texts = df['quote'].tolist()
quote_embeddings = model.encode(quote_texts, convert_to_numpy=True)

# Build FAISS index
index = faiss.IndexFlatIP(quote_embeddings.shape[1]) # IndexFlatIP for cosine similarity
index.add(quote_embeddings) # Add vectors to the index

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [25]:
def retrieve_similar_quotes(query, top_k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)
    return df.iloc[indices[0]]

In [26]:
# Use a summarization or text-generation model
generator = pipeline("text2text-generation", model="google/flan-t5-base")

def generate_rag_answer(query, top_k=5):
    # Retrieve relevant quotes
    relevant_df = retrieve_similar_quotes(query, top_k=top_k)
    context = " ".join(relevant_df['quote'].tolist())

    # Generate answer
    prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer:"
    response = generator(prompt, max_length=150, do_sample=True)[0]['generated_text']
    return response

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Device set to use cuda:0


In [31]:
question = "How can I stay motivated?"
answer = generate_rag_answer(question)
print("Answer:", answer)

Answer: Be where you are, stop over-thinking, and focus on what you are doing
